# Fragestellungen

1. In welchen Stadtteilen oder Regionen treten bestimmte Verbrechen häufiger auf?
    - Welche Stadtteile sind besonders von gewaltverbrechen, diebstahl, verkehrsdelikten betroffen?
    - Welche Art von Kriminalität hat in den Stadtteilen die letzten Jahren am stärksten zugenommen?
    - Gibt es saisonale unterschiede bei der Straftaten?
    - Gibt es eine Zunahme von Vandalismus während Feiertagen?
    - Sind Gewaltverbrechen häufiger an Wochenenden als an Wochentagen?
    - Hat die Corona-Pandemie die Anzahl bestimmter Verbrechen beeinflusst?
    - Welche Delikte werden häufiger in Verbindung mit anderen Straftaten begangen?

In [187]:
import pandas as pd
import plotly.express as px


In [188]:
df = pd.read_csv("../data/oh_encoded_categories.csv", index_col=0)

In [189]:
df.head(3)

,date,title,location,link,details,number,details_lemma,category,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,sonstige,hasskriminalität,gewaltverbrechen
0,2021-04-17 13:00:00,Brandsätze gegen Fassade geworfen - Gesuchter ...,Tempelhof-Schöneberg,/polizei/polizeimeldungen/pressemitteilung.100...,Im Zusammenhang mit dem im Oktober des vergang...,0841,Zusammenhang Oktober Jahr erfolgt Brandanschla...,vandalismus,False,True,False,False,False,False,False,False,False
1,2021-04-13 10:01:00,Jugendliche nach nächtlichem Überfall im Poliz...,Steglitz-Zehlendorf,/polizei/polizeimeldungen/2021/pressemitteilun...,Nach einem in der vergangenen Nacht gemeinscha...,0801,Nacht gemeinschaftlich begangen Raub jugendlic...,"gewaltverbrechen, diebstahl",False,False,False,False,False,True,False,False,True
2,2021-02-09 15:01:00,Betrüger und Trickdiebe als falsche Versicheru...,berlinweit,/polizei/polizeimeldungen/pressemitteilung.102...,Zwischenzeitlich konnte das Landeskriminalamt ...,0316,zwischenzeitlich Landeskriminalamt Rahmen inte...,betrug,False,False,False,False,True,False,False,False,False


In [190]:
df = df[~df["location"].str.contains(r"[0-9]", regex=True)]

In [191]:
df["location"].unique()

array(['Tempelhof-Schöneberg', 'Steglitz-Zehlendorf', 'berlinweit',
       'Mitte', 'Lichtenberg', 'Charlottenburg-Wilmersdorf', 'Neukölln',
       'Marzahn-Hellersdorf', 'Treptow-Köpenick',
       'Friedrichshain-Kreuzberg', 'Spandau', 'bezirksübergreifend',
       'Pankow', 'Reinickendorf', 'bundesweit'], dtype=object)

In [192]:
df["date"] = pd.to_datetime(df["date"])

In [193]:
df["tag_der_woche"] = df["date"].dt.day_of_week
df["tag_des_jahres"] = df["date"].dt.day_of_year
df["monat"] = df["date"].dt.month
df["jahr"] = df["date"].dt.year
df["tag_monat_jahr"] = df["date"].dt.date

In [194]:
loc_df = df.groupby(by='location').agg(verkehrsdelikte=('verkehrsdelikte', 'sum'),
                                       vandalismus=('vandalismus', 'sum'),
                                       sexualdelikte=('sexualdelikte', 'sum'),
                                       drogen=('drogen', 'sum'),
                                       betrug=('betrug', 'sum'),
                                       diebstahl=('diebstahl', 'sum'),
                                       hasskriminalität=('hasskriminalität', 'sum'),
                                       gewaltverbrechen=('gewaltverbrechen', 'sum'),
                                       sonstige=('sonstige', 'sum')
                                       )
 

In [195]:
loc_df["gesamt_straftaten"] = loc_df.sum(axis=1)
loc_df

,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige,gesamt_straftaten
location,,,,,,,,,,
Charlottenburg-Wilmersdorf,315,154,7,36,2,183,69,231,252,1249
Friedrichshain-Kreuzberg,185,237,18,79,2,159,116,480,271,1547
Lichtenberg,171,181,9,22,1,80,60,240,155,919
Marzahn-Hellersdorf,174,135,4,15,2,76,53,181,151,791
Mitte,334,264,15,67,4,246,166,611,377,2084
Neukölln,211,205,7,63,6,160,82,370,204,1308
Pankow,250,153,4,27,0,119,70,208,189,1020
Reinickendorf,209,116,3,11,2,80,42,171,131,765
Spandau,221,128,2,29,5,89,41,174,148,837


In [196]:
fig = px.bar(loc_df, 
             x=loc_df.index,
             y="gesamt_straftaten",
             title="Verteilung der Straftaten nach Standort",
             labels={"gesamt_straftaten": "Anzahl der Straftaten", "location": "Ort"}
             )

fig.show()

## Welche Stadtteile sind besonders von gewaltverbrechen, diebstahl, verkehrsdelikten betroffen?

In [202]:
fig = px.bar(loc_df, 
             x=loc_df.index,
             y=["gewaltverbrechen", "verkehrsdelikte", "diebstahl"],
             title="Verteilung der Straftaten nach Standort",
             labels={"value": "Anzahl der Straftaten", "location": "Ort"}
             )

fig.show()

In [197]:
month_df = df.groupby("tag_der_woche", as_index=False).agg(verkehrsdelikte=('verkehrsdelikte', 'sum'),
                                       vandalismus=('vandalismus', 'sum'),
                                       sexualdelikte=('sexualdelikte', 'sum'),
                                       drogen=('drogen', 'sum'),
                                       betrug=('betrug', 'sum'),
                                       diebstahl=('diebstahl', 'sum'),
                                       hasskriminalität=('hasskriminalität', 'sum'),
                                       gewaltverbrechen=('gewaltverbrechen', 'sum'),
                                       sonstige=('sonstige', 'sum')
                                       )

month_df

,tag_der_woche,verkehrsdelikte,vandalismus,sexualdelikte,drogen,betrug,diebstahl,hasskriminalität,gewaltverbrechen,sonstige
0,0,348,292,12,60,6,211,126,427,418
1,1,448,307,19,61,6,241,132,469,414
2,2,455,289,16,84,16,245,140,477,449
3,3,456,281,14,92,30,238,130,489,455
4,4,433,279,17,74,5,222,127,428,468
5,5,389,245,8,48,1,151,119,458,344
6,6,359,304,11,50,2,171,128,499,367


## Sind Gewaltverbrechen häufiger an Wochenenden als an Wochentagen?

In [199]:
fig = px.bar(month_df,
              x="tag_der_woche",
              y=["gewaltverbrechen"],
              labels={"value": "Anzahl der Gewaltverbrechen", "tag_der_woche": "Wochentag"})

fig.show()

## Welche Art von Kriminalität hat in den Stadtteilen die letzten Jahren am stärksten zugenommen?

In [218]:
trend_df = df.groupby("jahr", as_index=False).agg(verkehrsdelikte=('verkehrsdelikte', 'sum'),
                                       vandalismus=('vandalismus', 'sum'),
                                       sexualdelikte=('sexualdelikte', 'sum'),
                                       drogen=('drogen', 'sum'),
                                       betrug=('betrug', 'sum'),
                                       diebstahl=('diebstahl', 'sum'),
                                       hasskriminalität=('hasskriminalität', 'sum'),
                                       gewaltverbrechen=('gewaltverbrechen', 'sum'),
                                       sonstige=('sonstige', 'sum')
                                       )

trend_df = trend_df[trend_df["jahr"] < 2025]

In [222]:
fig = px.line(
    trend_df, 
    x="jahr", 
    y=["verkehrsdelikte", "vandalismus", "sexualdelikte", "drogen", "betrug", "diebstahl", "hasskriminalität", "gewaltverbrechen"], 
    title="Entwicklung der Gewaltverbrechen über die Jahre",
    labels={"jahr": "Jahr", "value": "Anzahl der Gewaltverbrechen"},
    markers=True
)

fig.update_layout(width=1200,
                  height=600)
fig.show()